#  LangChain with Conversation Memory
## Data Science Mentor App — Now with Memory!

###  What's New:
The mentor now **remembers your previous questions**.  
You can ask referential follow-ups like:
- *"What is overfitting?"*
- *"Can you give me an example of that?"* ← references previous answer
- *"How do I fix it?"* ← still references overfitting!

---
##  Step 1: Install Libraries

In [ ]:
!pip install langchain
!pip install langchain-groq
!pip install langchain-core

---
##  Step 2: Load Groq API Key

In [1]:
with open("groqapi.txt", "r") as f:
    api_key = f.read().strip()

print(" API Key loaded!")

 API Key loaded!


---
##  COMPONENT 1 — Prompt Template (with Memory Slot)

We use `MessagesPlaceholder` to inject the **full conversation history** into every prompt automatically.

In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Prompt Template with a placeholder for chat history
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are an expert Data Science Mentor with 10+ years of experience.
    Your job is to teach Data Science concepts clearly and simply.
    - Always give a clear explanation
    - Use a real-world example
    - End with one pro tip
    - Remember the full conversation and answer follow-up questions in context
    """),
    MessagesPlaceholder(variable_name="chat_history"),  # ← memory goes here
    ("human", "{question}")
])

print(" Prompt Template with Memory Slot created!")
print()
print(prompt)

 Prompt Template with Memory Slot created!

input_variables=['chat_history', 'question'] input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageC

---
## 🤖 COMPONENT 2 — Groq Model

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=api_key,
    temperature=0.7
)

print(" Groq Model Initialized!")
print(f"   Model : {model.model_name}")

##  COMPONENT 2 — Groq Model (LLaMA 3)

The **Model** is where the actual AI processing happens.  
We use **Groq** to run the open-source **LLaMA 3** model for free.

In [6]:
from langchain_groq import ChatGroq

# Initialize the Groq model
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=api_key,
    temperature=0.7
)

#  Preview the model config
print(" Groq Model Initialized!")
print()
print(f"  Model Name  : {model.model_name}")
print(f"  Temperature : {model.temperature}")

 Groq Model Initialized!

  Model Name  : llama-3.3-70b-versatile
  Temperature : 0.7


---
##  COMPONENT 3 — Output Parser

In [7]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

print(" Output Parser ready!")

 Output Parser ready!


---
##  COMPONENT 4 — Memory (Chat History)

`ChatMessageHistory` stores the full conversation.  
Every time you ask a question, we **add it to memory** and pass it into the prompt.

In [8]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# This stores the entire conversation in memory
memory = InMemoryChatMessageHistory()

print(" Memory (Chat History) initialized!")
print(f"   Messages stored so far: {len(memory.messages)}")

 Memory (Chat History) initialized!
   Messages stored so far: 0


---
##  Step 3: Build the Chain

In [9]:
# Connect all 3 components
chain = prompt | model | output_parser

print(" Chain built!")
print()
print("Pipeline:")
print("  Prompt Template (with memory slot)")
print("  |  Groq LLaMA 3")
print("  |  StrOutputParser")

 Chain built!

Pipeline:
  Prompt Template (with memory slot)
  |  Groq LLaMA 3
  |  StrOutputParser


---
##  Step 4: Helper Function — Ask with Memory

This function:
1. Takes your question
2. Passes the **full chat history** into the chain
3. Gets the answer
4. **Saves both** the question and answer back into memory

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

def ask_mentor(question: str) -> str:
    """Ask the Data Science Mentor. Memory is maintained automatically."""
    
    # Run the chain with full chat history
    response = chain.invoke({
        "question": question,
        "chat_history": memory.messages   # ← inject full history
    })
    
    # Save this turn into memory
    memory.add_message(HumanMessage(content=question))
    memory.add_message(AIMessage(content=response))
    
    return response


def show_memory():
    """Print the full conversation history stored in memory."""
    print(f"\n Memory — {len(memory.messages)} messages stored:")
    print("-" * 40)
    for i, msg in enumerate(memory.messages):
        role = " You" if isinstance(msg, HumanMessage) else " Mentor"
        print(f"[{i+1}] {role}: {msg.content[:100]}..." if len(msg.content) > 100 else f"[{i+1}] {role}: {msg.content}")
    print("-" * 40)


def clear_memory():
    """Reset the conversation memory."""
    memory.clear()
    print(" Memory cleared! Starting fresh conversation.")


print(" Helper functions ready: ask_mentor() | show_memory() | clear_memory()")

---
##  Step 5: Test Referential Questions

Watch how the mentor remembers context across questions!

In [ ]:
# ── Question 1: Ask a base topic ──
q1 = "What is overfitting in machine learning?"
print(f" You: {q1}")
print("=" * 50)
print(ask_mentor(q1))

In [ ]:
# ── Question 2: Referential follow-up ──
# Notice: we don't mention 'overfitting' — the model knows from memory!
q2 = "Can you give me a real code example of that?"
print(f" You: {q2}")
print("=" * 50)
print(ask_mentor(q2))

In [ ]:
# ── Question 3: Another follow-up ──
q3 = "How do I fix this problem?"
print(f" You: {q3}")
print("=" * 50)
print(ask_mentor(q3))

In [ ]:
# ── Question 4: Even more referential ──
q4 = "Which of those techniques is the easiest to implement for a beginner?"
print(f" You: {q4}")
print("=" * 50)
print(ask_mentor(q4))

In [ ]:
#  See all messages stored in memory
show_memory()

---
##  Step 6: Interactive Chat App with Memory

In [ ]:
# Clear memory before starting fresh interactive session
clear_memory()

print(" Data Science Mentor App — with Memory!")
print("Commands: 'history' = show memory | 'clear' = reset | 'exit' = quit")
print()

while True:
    user_input = input(" You: ").strip()
    
    if user_input.lower() == "exit":
        print(" Goodbye! Keep learning!")
        break
    elif user_input.lower() == "history":
        show_memory()
        continue
    elif user_input.lower() == "clear":
        clear_memory()
        continue
    elif user_input == "":
        print("  Please type a question.")
        continue
    
    print("\n Mentor: (thinking...)\n")
    answer = ask_mentor(user_input)
    print(f" Mentor: {answer}")
    print()

---
##  Full Summary

```
┌──────────────────────────────────┐
│  COMPONENT 1: Prompt Template    │
│  System Prompt (DS Mentor)       │
│  + MessagesPlaceholder           │  ← chat_history injected here
│  + {question}                    │
└────────────┬─────────────────────┘
             │
┌────────────▼─────────────────────┐
│  COMPONENT 2: Groq Model         │
│  LLaMA 3.3 70B                   │
└────────────┬─────────────────────┘
             │
┌────────────▼─────────────────────┐
│  COMPONENT 3: Output Parser      │
│  StrOutputParser → clean string  │
└────────────┬─────────────────────┘
             │
┌────────────▼─────────────────────┐
│  COMPONENT 4: Memory             │
│  InMemoryChatMessageHistory      │  ← saves every Q&A
│  HumanMessage + AIMessage        │  ← fed back to prompt
└──────────────────────────────────┘
```

| Component | Class | Purpose |
|-----------|-------|---------|
| Prompt Template | `ChatPromptTemplate` + `MessagesPlaceholder` | Holds system prompt + memory slot |
| Model | `ChatGroq` | LLaMA 3.3 70B via Groq |
| Output Parser | `StrOutputParser` | Clean string output |
| Memory | `InMemoryChatMessageHistory` | Stores full conversation for referential Q&A |

###  Example Referential Flow:
```
You    → "What is overfitting?"
Mentor → [explains overfitting]

You    → "Can you give an example of that?"   ← 'that' = overfitting (from memory)
Mentor → [gives overfitting example]

You    → "How do I fix it?"                   ← 'it' = overfitting (from memory)
Mentor → [explains fixes for overfitting]
```